# Stabilizer Formalism in Quantum Computing
### A First-Principles Introduction
---
**What this notebook covers:**
- The Pauli group and its algebraic structure
- What it means for a state to be 'stabilized'
- Stabilizer states and their tableau representation
- Clifford gates and how they act on stabilizers
- Measurement in the stabilizer formalism
- The Gottesman-Knill theorem
- Application to quantum error correction
- Magic states and the limits of stabilizer methods

> **Prerequisites:** Linear algebra, basic quantum mechanics (bra-ket notation, qubits, unitary gates), familiarity with Python.

In [ ]:
# Section 0: Setup and imports
import time
import warnings
import numpy as np
import matplotlib.pyplot as plt

try:
    from qiskit import QuantumCircuit
    from qiskit.quantum_info import Statevector, Pauli
    from qiskit_aer import Aer
    QISKIT_AVAILABLE = True
except Exception as exc:
    QISKIT_AVAILABLE = False
    warnings.warn(f"Qiskit unavailable: {exc}")

print("numpy:", np.__version__)
if QISKIT_AVAILABLE:
    import qiskit
    import qiskit_aer
    print("qiskit:", qiskit.__version__)
    print("qiskit-aer:", qiskit_aer.__version__)
else:
    print("qiskit: unavailable")

## 1. Why Stabilizer Formalism Exists

Simulating an $n$-qubit quantum state naively requires storing $2^n$ complex amplitudes.

| Qubits | Amplitudes | Memory (complex128) |
|--------|------------|---------------------|
| 10 | 1,024 | ~16 KB |
| 20 | 1,048,576 | ~16 MB |
| 30 | ~10^9 | ~16 GB |
| 50 | ~10^15 | ~16 PB |

The stabilizer formalism (Gottesman, 1997) identifies a structured subset of quantum states — **stabilizer states** — that can be described with only $O(n^2)$ classical bits. The key theorem:

> **Gottesman-Knill Theorem:** Any quantum circuit consisting of Clifford gates, computational basis preparations, and Pauli measurements can be efficiently simulated classically in $O(n^2)$ space and $O(n^2)$ time per operation.

This is not because these circuits are 'not quantum' — they create entanglement and superposition. It is because their structure is algebraically tractable.

## 2. The Pauli Group

### 2.1 Single-Qubit Paulis

The four single-qubit Pauli matrices are:

$$I = \begin{pmatrix}1&0\\0&1\end{pmatrix}, \quad X = \begin{pmatrix}0&1\\1&0\end{pmatrix}, \quad Y = \begin{pmatrix}0&-i\\i&0\end{pmatrix}, \quad Z = \begin{pmatrix}1&0\\0&-1\end{pmatrix}$$

Key properties:
- Each is **Hermitian**: $P^\dagger = P$
- Each squares to identity: $P^2 = I$
- Each has eigenvalues $\{+1, -1\}$
- Any two distinct Paulis **anticommute**: $\{X, Z\} = XZ + ZX = 0$

### 2.2 The $n$-Qubit Pauli Group

The $n$-qubit Pauli group $\mathcal{P}_n$ consists of all $n$-fold tensor products of single-qubit Paulis, with phases from $\{\pm 1, \pm i\}$:

$$\mathcal{P}_n = \{ i^k \, P_1 \otimes P_2 \otimes \cdots \otimes P_n \mid k \in \{0,1,2,3\},\ P_j \in \{I, X, Y, Z\} \}$$

Size: $|\mathcal{P}_n| = 4^{n+1}$.

### 2.3 Commutation Rule

Two $n$-qubit Paulis $P, Q \in \mathcal{P}_n$ either commute ($PQ = QP$) or anticommute ($PQ = -QP$). They anticommute if and only if the number of qubit positions where they have **anticommuting single-qubit factors** is **odd**.

In [ ]:
# Section 2: Pauli matrices and properties
I = np.array([[1, 0], [0, 1]], dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
paulis = {"I": I, "X": X, "Y": Y, "Z": Z}

def commutes(P, Q, atol=1e-10):
    return np.allclose(P @ Q, Q @ P, atol=atol)

for name, P in paulis.items():
    print(f"{name} =\n{P}\n")
    print(f"{name}^2 == I:", np.allclose(P @ P, I))
    print(f"eigenvalues({name}):", np.round(np.linalg.eigvalsh(P), 8), "\n")

print("XZ = -ZX:", np.allclose(X @ Z, -Z @ X))
print("XY = iZ:", np.allclose(X @ Y, 1j * Z))
print("YZ = iX:", np.allclose(Y @ Z, 1j * X))
print("ZX = iY:", np.allclose(Z @ X, 1j * Y))

In [ ]:
# Section 2: Pauli multiplication table visualization
label_order = ["I", "X", "Y", "Z"]

def multiply_pauli(a, b):
    phases = {
        ("I", "I"): (1, "I"), ("I", "X"): (1, "X"), ("I", "Y"): (1, "Y"), ("I", "Z"): (1, "Z"),
        ("X", "I"): (1, "X"), ("X", "X"): (1, "I"), ("X", "Y"): (1j, "Z"), ("X", "Z"): (-1j, "Y"),
        ("Y", "I"): (1, "Y"), ("Y", "X"): (-1j, "Z"), ("Y", "Y"): (1, "I"), ("Y", "Z"): (1j, "X"),
        ("Z", "I"): (1, "Z"), ("Z", "X"): (1j, "Y"), ("Z", "Y"): (-1j, "X"), ("Z", "Z"): (1, "I"),
    }
    return phases[(a, b)]

colors = {"I": "white", "X": "#cfe8ff", "Y": "#dff5df", "Z": "#ffe8cc"}
cell_text, cell_colors = [], []
for r in label_order:
    trow, crow = [], []
    for c in label_order:
        ph, out = multiply_pauli(r, c)
        phs = {1: "", -1: "-", 1j: "i", -1j: "-i"}[ph]
        trow.append(f"{phs}{out}" if phs else out)
        crow.append(colors[out])
    cell_text.append(trow)
    cell_colors.append(crow)

fig, ax = plt.subplots(figsize=(6, 3))
ax.axis("off")
tbl = ax.table(cellText=cell_text, rowLabels=label_order, colLabels=label_order, cellLoc="center", loc="center", cellColours=cell_colors)
tbl.scale(1.2, 1.6)
ax.set_title("Single-Qubit Pauli Multiplication Table")
plt.tight_layout()
plt.show()

## 3. Stabilizer States

### 3.1 What It Means to Be Stabilized

A non-zero state $|\psi\rangle$ is **stabilized** by operator $P$ if:
$$P|\psi\rangle = |\psi\rangle$$
That is, $|\psi\rangle$ is a $+1$ eigenstate of $P$.

**Examples:**
- $Z|0\rangle = |0\rangle$ — so $|0\rangle$ is stabilized by $Z$
- $X|{+}\rangle = |{+}\rangle$ — so $|{+}\rangle$ is stabilized by $X$
- $Z|1\rangle = -|1\rangle$ — so $|1\rangle$ is stabilized by $-Z$

### 3.2 The Stabilizer Group

The **stabilizer group** of $|\psi\rangle$ is:
$$\mathcal{S}(|\psi\rangle) = \{ P \in \mathcal{P}_n \mid P|\psi\rangle = |\psi\rangle \}$$

This is always a group (closed under multiplication, contains $I$). For an $n$-qubit stabilizer state, $\mathcal{S}$ is generated by exactly $n$ **independent, mutually commuting** Pauli operators that do not include $-I$.

### 3.3 Why Generators Must Commute

If $A|\psi\rangle = |\psi\rangle$ and $B|\psi\rangle = |\psi\rangle$, then $AB|\psi\rangle = A|\psi\rangle = |\psi\rangle$. But if $A$ and $B$ anticommuted — $AB = -BA$ — then:
$$AB|\psi\rangle = |\psi\rangle \implies BA|\psi\rangle = -|\psi\rangle$$
which contradicts $A|\psi\rangle = |\psi\rangle$ applied after $B$. So generators **must commute**.

### 3.4 Uniqueness

For $n$ independent commuting Pauli generators (not containing $-I$), the $+1$ joint eigenspace has dimension exactly $2^{n-k}$ where $k$ is the number of independent generators. With exactly $n$ generators for $n$ qubits, the eigenspace is 1-dimensional — a unique state.

In [ ]:
# Section 3: Verify stabilizer examples numerically
ket_0 = np.array([1, 0], dtype=complex)
ket_1 = np.array([0, 1], dtype=complex)
ket_plus = np.array([1, 1], dtype=complex) / np.sqrt(2)
ket_minus = np.array([1, -1], dtype=complex) / np.sqrt(2)
ket_plus_i = np.array([1, 1j], dtype=complex) / np.sqrt(2)
ket_minus_i = np.array([1, -1j], dtype=complex) / np.sqrt(2)

def is_stabilized(state, op, atol=1e-10):
    return np.allclose(op @ state, state, atol=atol)

tests = [
    ("|0>", ket_0, "+Z", Z),
    ("|1>", ket_1, "-Z", -Z),
    ("|+>", ket_plus, "+X", X),
    ("|->", ket_minus, "-X", -X),
    ("|+i>", ket_plus_i, "+Y", Y),
    ("|-i>", ket_minus_i, "-Y", -Y),
]
print(f"{'State':<6} {'Expected':<8} {'Stabilized?':<12}")
for name, st, exp, op in tests:
    print(f"{name:<6} {exp:<8} {str(is_stabilized(st, op)):<12}")
print("\nX@Z == -Z@X:", np.allclose(X @ Z, -Z @ X))

In [ ]:
# Section 3: Bloch sphere of stabilizer states
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
u = np.linspace(0, 2*np.pi, 60)
v = np.linspace(0, np.pi, 30)
xs = np.outer(np.cos(u), np.sin(v))
ys = np.outer(np.sin(u), np.sin(v))
zs = np.outer(np.ones_like(u), np.cos(v))
ax.plot_wireframe(xs, ys, zs, color='lightgray', linewidth=0.4)
pts = {'|0>': (0,0,1,'red'), '|1>': (0,0,-1,'red'), '|+>': (1,0,0,'blue'), '|->': (-1,0,0,'blue'), '|+i>': (0,1,0,'green'), '|-i>': (0,-1,0,'green')}
for lbl, (x,y,z,c) in pts.items():
    ax.scatter([x],[y],[z], color=c, s=60)
    ax.plot([0,x],[0,y],[0,z], '--', color=c, alpha=0.6)
    ax.text(x*1.1,y*1.1,z*1.1,lbl)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('The 6 Single-Qubit Stabilizer States on the Bloch Sphere')
ax.set_box_aspect((1,1,1))
plt.tight_layout()
plt.show()

## 4. The Stabilizer Tableau

### 4.1 Binary Representation of Paulis

Every $n$-qubit Pauli can be encoded as a binary vector of length $2n$ plus a sign bit:

$$P = i^s \cdot X^{x_1}Z^{z_1} \otimes X^{x_2}Z^{z_2} \otimes \cdots \otimes X^{x_n}Z^{z_n}$$

where $x_j, z_j \in \{0,1\}$. The encoding:

| $x_j$ | $z_j$ | Pauli on qubit $j$ |
|--------|--------|--------------------|
| 0 | 0 | $I$ |
| 1 | 0 | $X$ |
| 0 | 1 | $Z$ |
| 1 | 1 | $Y$ |

A generator $g_i$ is stored as a row: $[x_{i,1}, \ldots, x_{i,n} \mid z_{i,1}, \ldots, z_{i,n} \mid r_i]$ where $r_i \in \{0,1\}$ encodes sign ($0 \to +1$, $1 \to -1$).

### 4.2 The Full Tableau

For $n$ qubits, the stabilizer tableau is a $(2n \times (2n+1))$ binary matrix:
- **Rows 1 to $n$:** Destabilizer generators $\bar{g}_1, \ldots, \bar{g}_n$
- **Rows $n+1$ to $2n$:** Stabilizer generators $g_1, \ldots, g_n$
- **Last column:** Sign bits

The initial tableau for $|0\rangle^{\otimes n}$ has destabilizers $X_i$ and stabilizers $Z_i$ for each qubit $i$.

In [ ]:
# Section 4-6: StabilizerTableau class
class StabilizerTableau:
    def __init__(self, n):
        self.n = int(n)
        self.tableau = np.zeros((2*self.n, 2*self.n + 1), dtype=np.uint8)
        for i in range(self.n):
            self.tableau[i, i] = 1
            self.tableau[self.n + i, self.n + i] = 1

    @property
    def x_block(self): return self.tableau[:, :self.n]
    @property
    def z_block(self): return self.tableau[:, self.n:2*self.n]
    @property
    def signs(self): return self.tableau[:, 2*self.n]

    def row_to_label(self, row):
        sign = '-' if self.signs[row] else '+'
        s = []
        for q in range(self.n):
            x, z = self.x_block[row,q], self.z_block[row,q]
            s.append('I' if (x,z)==(0,0) else 'X' if (x,z)==(1,0) else 'Z' if (x,z)==(0,1) else 'Y')
        return sign + ''.join(s)

    def state_str(self): return [self.row_to_label(self.n+i) for i in range(self.n)]

    def display(self):
        hdr = [f"x{i+1}" for i in range(self.n)] + [f"z{i+1}" for i in range(self.n)] + ["r", "label"]
        print(' '.join(f"{h:>3}" for h in hdr))
        for i in range(2*self.n):
            k = 'D' if i < self.n else 'S'
            row = list(self.tableau[i,:])
            print(f"{k}{(i%self.n)+1}", ' '.join(f"{v:>3}" for v in row), f"{self.row_to_label(i):>8}")

    def rowmul(self, i, j):
        self.signs[i] ^= self.signs[j]
        self.x_block[i,:] ^= self.x_block[j,:]
        self.z_block[i,:] ^= self.z_block[j,:]

    def apply_h(self, q):
        for i in range(2*self.n):
            if self.x_block[i,q] and self.z_block[i,q]:
                self.signs[i] ^= 1
            self.x_block[i,q], self.z_block[i,q] = self.z_block[i,q], self.x_block[i,q]

    def apply_s(self, q):
        for i in range(2*self.n):
            if self.x_block[i,q] and self.z_block[i,q]:
                self.signs[i] ^= 1
            self.z_block[i,q] ^= self.x_block[i,q]

    def apply_cnot(self, c, t):
        for i in range(2*self.n):
            self.signs[i] ^= self.x_block[i,c] & self.z_block[i,t] & (self.x_block[i,t] ^ self.z_block[i,c] ^ 1)
            self.x_block[i,t] ^= self.x_block[i,c]
            self.z_block[i,c] ^= self.z_block[i,t]

    def measure_z(self, q, rng=np.random.default_rng()):
        p = next((i for i in range(self.n,2*self.n) if self.x_block[i,q]), None)
        if p is None:
            val = 0
            for i in range(self.n):
                if self.x_block[i,q]:
                    val ^= self.signs[i]
            return int(val)
        b = int(rng.integers(0,2))
        for i in range(2*self.n):
            if i != p and self.x_block[i,q]:
                self.rowmul(i,p)
        self.tableau[p-self.n,:] = self.tableau[p,:]
        self.tableau[p,:] = 0
        self.z_block[p,q] = 1
        self.signs[p] = b
        return b

tb = StabilizerTableau(3)
tb.display()
print("Initial stabilizers:", tb.state_str())

## 5. Clifford Gates

### 5.1 The Clifford Group

A unitary $U$ is a **Clifford gate** if it maps the Pauli group to itself under conjugation:
$$\forall P \in \mathcal{P}_n: \quad U P U^\dagger \in \mathcal{P}_n$$

This is the key property: Cliffords **permute** the Paulis. So if $P|\psi\rangle = |\psi\rangle$, then $(UPU^\dagger)(U|\psi\rangle) = U|\psi\rangle$. The new state $U|\psi\rangle$ is stabilized by $UPU^\dagger$.

**Consequence for simulation:** To apply $U$ to a stabilizer state, replace each generator $g_i \to U g_i U^\dagger$. No matrix exponentiation — just conjugation rules.

### 5.2 Single-Qubit Clifford Conjugation Rules

$$H X H^\dagger = Z, \quad H Z H^\dagger = X, \quad H Y H^\dagger = -Y$$
$$S X S^\dagger = Y, \quad S Z S^\dagger = Z, \quad S Y S^\dagger = -X$$

### 5.3 CNOT Conjugation Rules
(control qubit $c$, target qubit $t$)

$$\text{CNOT}: \quad X_c I_t \to X_c X_t, \quad I_c X_t \to I_c X_t$$
$$\text{CNOT}: \quad Z_c I_t \to Z_c I_t, \quad I_c Z_t \to Z_c Z_t$$

These four rules, extended by linearity across qubit positions, fully define CNOT's action on any Pauli.

In [ ]:
# Section 5: Verify Clifford conjugation rules
H = np.array([[1,1],[1,-1]], dtype=complex) / np.sqrt(2)
S = np.array([[1,0],[0,1j]], dtype=complex)

def conjugate(U, P): return U @ P @ U.conj().T
for lbl, lhs, rhs in [
    ("H X H† = Z", conjugate(H, X), Z),
    ("H Z H† = X", conjugate(H, Z), X),
    ("H Y H† = -Y", conjugate(H, Y), -Y),
    ("S X S† = Y", conjugate(S, X), Y),
    ("S Z S† = Z", conjugate(S, Z), Z),
    ("S Y S† = -X", conjugate(S, Y), -X),
]:
    print(lbl, "->", np.allclose(lhs, rhs))

In [ ]:
# Section 5: Tableau gate demo
t1 = StabilizerTableau(1)
print("Start |0>:")
t1.display()
t1.apply_h(0)
print("After H -> |+>:")
t1.display()
print("Stabilizers:", t1.state_str())

bell = StabilizerTableau(2)
bell.apply_h(0)
bell.apply_cnot(0,1)
print("\nBell state tableau:")
bell.display()
print("Stabilizers:", bell.state_str())

## 6. Measurement in the Stabilizer Formalism

Measuring Pauli $M$ on a stabilizer state with generators $\{g_1, \ldots, g_n\}$ has two cases:

### Case 1: $M$ Commutes With All Generators

The measurement outcome is **deterministic**. The post-measurement state is unchanged. To find the outcome:
- Extend the stabilizer generators to the full group (closure under multiplication)
- If $M \in \mathcal{S}$, outcome is $+1$
- If $-M \in \mathcal{S}$, outcome is $-1$

In the tableau algorithm: use the destabilizer rows to determine the outcome via a sequence of row multiplications.

### Case 2: $M$ Anticommutes With at Least One Generator

The outcome is **random** ($+1$ or $-1$, each with probability $\frac{1}{2}$). After measurement with outcome $(-1)^b$:
1. Let $g_p$ be any generator anticommuting with $M$ (the pivot)
2. For every other generator $g_i$ anticommuting with $M$: replace $g_i \leftarrow g_i \cdot g_p$
3. Replace $g_p \leftarrow (-1)^b \cdot M$
4. Update destabilizers analogously

Cost: $O(n)$ row operations → $O(n^2)$ total. No exponential overhead.

### Key Insight

Measurement in the stabilizer formalism is a **tableau surgery operation**: it never requires computing the full wavefunction.

In [ ]:
# Section 6: Measurement demo
rng = np.random.default_rng(7)
outcomes = []
for _ in range(10):
    b = StabilizerTableau(2)
    b.apply_h(0)
    b.apply_cnot(0,1)
    outcomes.append(b.measure_z(0, rng=rng))
print("Bell-state Z measurement on qubit 0 (10 trials):", outcomes)
psi = np.array([1,0,0,1], dtype=complex) / np.sqrt(2)
ZZ = np.kron(Z,Z)
print("<ZZ> for |Phi+>:", np.vdot(psi, ZZ @ psi).real)

## 7. The Gottesman-Knill Theorem

**Theorem (Gottesman 1997, Knill):** The following operations can be efficiently simulated on a classical computer:
1. Preparation of qubits in computational basis states
2. Application of Clifford gates: $H$, $S$, $\text{CNOT}$
3. Measurement of Pauli observables

**Simulation cost:** $O(n^2)$ bits of memory, $O(n^2)$ time per operation.

### What Clifford Circuits Can Do
| Capability | Clifford Circuits |
|-----------|-------------------|
| Create superposition | ✓ ($H|0\rangle = |{+}\rangle$) |
| Create entanglement | ✓ (Bell states via $H$ + CNOT) |
| Achieve quantum parallelism | ✓ |
| **Universal quantum computation** | ✗ |
| Classical efficient simulation | ✓ |

### What Breaks the Classicality

Adding the **T gate** ($T = \text{diag}(1, e^{i\pi/4})$) to the Clifford group yields a **universal** gate set. The T gate takes:
$$T X T^\dagger = \frac{X + Y}{\sqrt{2}}$$
This is outside the Pauli group — T is not Clifford. The resource that T injects is called **magic**.

In [ ]:
# Section 7: Benchmark tableau
rng = np.random.default_rng(42)
def random_clifford_circuit(n, depth):
    ops = []
    for _ in range(depth):
        g = rng.choice(["h", "s", "cnot"])
        if g in ("h", "s"):
            ops.append((g, int(rng.integers(0, n))))
        else:
            c = int(rng.integers(0, n))
            t = int(rng.integers(0, n-1))
            t += (t >= c)
            ops.append(("cnot", c, t))
    return ops
def simulate_tableau(n, ops):
    t = StabilizerTableau(n)
    for op in ops:
        if op[0] == 'h': t.apply_h(op[1])
        elif op[0] == 's': t.apply_s(op[1])
        else: t.apply_cnot(op[1], op[2])
    return t
ns = [2,4,8,16,32]
depth = 50
tab = []
for n in ns:
    times = []
    for _ in range(5):
        ops = random_clifford_circuit(n, depth)
        t0 = time.perf_counter()
        simulate_tableau(n, ops)
        t1 = time.perf_counter()
        times.append((t1-t0)*1000)
    tab.append(np.median(times))
plt.figure(figsize=(7,4))
plt.plot(ns, tab, 'o-', label='Tableau')
plt.yscale('log')
plt.xlabel('Number of qubits (n)')
plt.ylabel('Median simulation time (ms)')
plt.title('Classical Simulation Time: Stabilizer Tableau vs Statevector')
plt.legend()
plt.tight_layout()
plt.show()
print(dict(zip(ns, np.round(tab,3))))

## 8. Quantum Error Correction with Stabilizer Codes

### 8.1 The Stabilizer Code Framework

A **stabilizer code** $[[n, k, d]]$ encodes $k$ logical qubits into $n$ physical qubits with distance $d$:
- The **codespace** is the $+1$ eigenspace of an abelian group $\mathcal{S}$ of $n-k$ independent Pauli generators called **check operators** (or stabilizers)
- **Logical operators** $\bar{X}_i$, $\bar{Z}_i$ act on the encoded qubits and commute with $\mathcal{S}$ but are not in $\mathcal{S}$
- An **error** $E$ is detectable if $E$ anticommutes with at least one element of $\mathcal{S}$
- The **syndrome** is the tuple of $\pm 1$ measurement outcomes of the check operators

### 8.2 Error Detection Mechanism

$$|\psi_L\rangle \xrightarrow{\text{error } E} E|\psi_L\rangle \xrightarrow{\text{measure } g_i} \begin{cases} +1 & \text{if } [E, g_i] = 0 \\ -1 & \text{if } \{E, g_i\} = 0 \end{cases}$$

The pattern of $-1$ outcomes (the syndrome) identifies the error — without measuring the logical qubit itself, which would collapse the encoded information.

In [ ]:
# Section 8: 3-qubit bit-flip code syndrome table
syndrome_to_qubit = {(+1,+1): None, (-1,+1): 0, (-1,-1): 1, (+1,-1): 2}
def bitflip_syndrome(error_qubit=None):
    e = [0,0,0]
    if error_qubit is not None:
        e[error_qubit] = 1
    s1 = -1 if (e[0] ^ e[1]) else +1
    s2 = -1 if (e[1] ^ e[2]) else +1
    return s1, s2
print(f"{'Injected X error':<20}{'Syndrome (ZZI,IZZ)':<22}{'Correction':<12}")
for eq in [None,0,1,2]:
    s = bitflip_syndrome(eq)
    corr = syndrome_to_qubit[s]
    print(f"{('None' if eq is None else f'X on q{eq}'):<20}{str(s):<22}{('None' if corr is None else f'X on q{corr}'):<12}")

In [ ]:
# Section 8: Steane [[7,1,3]] code commutation checks
stab = ["IIIXXXX", "IXXIIXX", "XIXIXIX", "IIIZZZZ", "IZZIIZZ", "ZIZIZIZ"]
XL, ZL = "XXXXXXX", "ZZZZZZZ"
anticomm = {('X','Z'),('Z','X'),('X','Y'),('Y','X'),('Y','Z'),('Z','Y')}
def commute(p, q): return (sum((a,b) in anticomm for a,b in zip(p,q)) % 2) == 0
m = np.array([[1 if commute(a,b) else 0 for b in stab] for a in stab])
print("6x6 commutation matrix (1=commute):")
print(m)
for i,g in enumerate(stab,1):
    print(f"g{i}: commute(XL,g{i})={commute(XL,g)} commute(ZL,g{i})={commute(ZL,g)}")
print("X_L and Z_L anticommute:", not commute(XL, ZL))

## 9. Beyond Stabilizers: Magic States and the T Gate

### 9.1 The T Gate Breaks Stabilizer Structure

The T gate is:
$$T = \begin{pmatrix}1 & 0 \\ 0 & e^{i\pi/4}\end{pmatrix}$$

Under conjugation:
$$T X T^\dagger = \frac{X + Y}{\sqrt{2}} \notin \mathcal{P}_1$$

So T is **not Clifford**. Applying T to a stabilizer state produces a **non-stabilizer state** (generically).

### 9.2 Magic States

A **magic state** is any state that cannot be expressed as a stabilizer state. The prototypical example is the T-type magic state:
$$|T\rangle = T|{+}\rangle = \frac{|0\rangle + e^{i\pi/4}|1\rangle}{\sqrt{2}}$$

This state is not stabilized by any Pauli (its Bloch vector points to $(\frac{1}{\sqrt{2}}, \frac{1}{\sqrt{2}}, 0) \cdot \frac{1}{\sqrt{2}}$... wait, more precisely it lies off all Pauli axes).

### 9.3 Why T-Count Matters

In fault-tolerant quantum computing, Clifford gates are cheap (transversal in most codes). T gates require **magic state distillation** — a costly purification protocol. As a result:
- T-count is the dominant cost metric in fault-tolerant resource estimation
- Reducing T-count is an active research area
- Classical simulation cost scales exponentially in T-count (not circuit depth)

In [ ]:
# Section 9: Magic state demo
T = np.diag([1, np.exp(1j*np.pi/4)])
ket_T = T @ ket_plus
def find_stabilizer(state, atol=1e-8):
    cands = {"+X": X, "-X": -X, "+Y": Y, "-Y": -Y, "+Z": Z, "-Z": -Z}
    return [k for k,P in cands.items() if np.allclose(P @ state, state, atol=atol)]
for n,s in [("|0>",ket_0),("|1>",ket_1),("|+>",ket_plus),("|->",ket_minus),("|T>",ket_T)]:
    print(n, "->", find_stabilizer(s))
TXTd = T @ X @ T.conj().T
is_pauli = any(np.allclose(TXTd, c * P) for c in [1,-1,1j,-1j] for P in [I,X,Y,Z])
print("T X T† is Pauli up to phase:", is_pauli)

In [ ]:
# Section 9: Clifford+T circuit demo
if QISKIT_AVAILABLE:
    qc = QuantumCircuit(3)
    qc.h(0); qc.cx(0,1); qc.s(2); qc.h(2); qc.cx(1,2)
    qc.measure_all()
    backend = Aer.get_backend('aer_simulator')
    counts = backend.run(qc, shots=1000).result().get_counts()
    print("Qiskit counts:", counts)
else:
    print("Qiskit unavailable. Skipping Aer simulation.")

t = StabilizerTableau(3)
ops = [("h",0), ("cnot",0,1), ("s",2), ("h",2), ("cnot",1,2)]
for op in ops:
    if op[0] == "h": t.apply_h(op[1])
    elif op[0] == "s": t.apply_s(op[1])
    else: t.apply_cnot(op[1], op[2])
    print(op, "->", t.state_str())

## 10. Conceptual Summary

### The Big Picture

```
All n-qubit quantum states (2^n complex amplitudes)
        |
        ├── Stabilizer states (O(n²) bits)
        │         |
        │         ├── Clifford circuits preserve this set
        │         ├── Pauli measurements stay efficient
        │         └── Gottesman-Knill: classically simulable
        │
        └── Non-stabilizer states (magic states)
                  |
                  ├── Generated by T gate applied to stabilizer states
                  ├── Required for universal quantum computation
                  └── T-count measures simulation hardness
```

### Reference Table: Stabilizer Formalism in One Page

| Concept | Definition | Why It Matters |
|---------|-----------|----------------|
| Pauli group $\mathcal{P}_n$ | $n$-fold Pauli tensor products ± phase | Algebraic foundation |
| Stabilizer group | Abelian Pauli subgroup fixing $|\psi\rangle$ | Compact state encoding |
| Stabilizer state | Uniquely fixed by $n$ independent Paulis | $O(n^2)$ classical representation |
| Clifford gate | Unitary normalizing $\mathcal{P}_n$ | Tableau-updatable |
| Destabilizer | Conjugate generators | Enables deterministic measurement |
| Stabilizer tableau | $2n \times (2n+1)$ binary matrix | The simulation data structure |
| Gottesman-Knill | Clifford circuits $\Rightarrow$ poly simulation | Core efficiency theorem |
| Stabilizer code | Codespace = $+1$ eigenspace of checks | Quantum error correction |
| Magic state | Any non-stabilizer state | Resource for universality |
| T-count | Number of T gates in circuit | Fault-tolerant resource cost |

### Further Reading
- Gottesman (1997) — *Stabilizer Codes and Quantum Error Correction* (PhD thesis, Caltech)
- Aaronson & Gottesman (2004) — *Improved simulation of stabilizer circuits*, PRA 70, 052328
- Nielsen & Chuang (2000) — *Quantum Computation and Quantum Information*, Chapter 10
- Gottesman (2009) — *Introduction to QEC and Fault-Tolerant QC*, arXiv:0904.2557
- Gidney (2021) — *Stim: A fast stabilizer circuit simulator*, Quantum 5, 497

In [ ]:
# Section 10: GHZ demo
ops = [("H",0), ("CNOT",0,1), ("CNOT",0,2)]
tg = StabilizerTableau(3)
for op in ops:
    if op[0] == "H": tg.apply_h(op[1])
    else: tg.apply_cnot(op[1], op[2])
    print(op, "stabilizers:", tg.state_str())
rng = np.random.default_rng(123)
meas = []
for _ in range(20):
    g = StabilizerTableau(3)
    g.apply_h(0); g.apply_cnot(0,1); g.apply_cnot(0,2)
    meas.append(g.measure_z(0, rng=rng))
vals, counts = np.unique(meas, return_counts=True)
plt.figure(figsize=(5,3))
plt.bar([str(v) for v in vals], counts, color=['#77aaff','#ff9999'])
plt.xlabel('Measurement outcome b on qubit 0')
plt.ylabel('Count over 20 runs')
plt.title('GHZ: random Z measurement on qubit 0')
plt.tight_layout()
plt.show()
print("Outcomes:", meas)